# 02 – Làm sạch và chia train/test

**Đầu vào:** 8 file CSV trong `data/MachineLearningCVE/`.
**Đầu ra:** `data/processed/train.pkl`, `test.pkl` (68 đặc trưng + `Label`, 9 lớp). Mọi thí nghiệm sau dùng chung bản chia này.

1. Đọc và gộp 8 file
2. Gộp 15 nhãn thành 9 lớp
3. Bỏ cột không dùng
4. Xóa dòng có NaN, Infinity
5. Xóa dòng trùng
6. Bỏ dòng mâu thuẫn nhãn
7. Giữ ngẫu nhiên 20% BENIGN
8. Chia train/test 80/20, phân tầng
9. Kiểm tra
10. Lưu

Mọi bước loại dòng đều làm trước khi chia, để không flow nào có mặt ở cả train lẫn test. Chuẩn hóa và resample không làm ở đây mà nằm trong Pipeline (notebook 03–04), để chỉ học từ tập train.

Chạy dưới 1 phút, cần khoảng 4 GB RAM.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "MachineLearningCVE"
OUT_DIR = DATA_DIR / "processed"
RANDOM_STATE = 42  # cố định để chạy lại lúc nào cũng ra đúng kết quả này

# Ghi lại số mẫu của từng lớp sau mỗi bước, cuối notebook in thành một bảng cho báo cáo
class_counts = {}

## 1. Đọc và gộp 8 file

Cùng cách đọc với notebook 01. Đổi sang `float32` ngay từ đầu cũng là điều kiện để bước 5 xóa trùng đúng.

In [2]:
frames = []
for f in sorted(RAW_DIR.glob("*.csv")):
    part = pd.read_csv(f, encoding="latin-1", low_memory=False)
    part.columns = part.columns.str.strip()
    float_cols = part.select_dtypes("float64").columns
    part[float_cols] = part[float_cols].astype("float32")
    frames.append(part)
    print(f"Read {f.name}: {len(part):,} rows")

df = pd.concat(frames, ignore_index=True)
del frames
df["Label"] = df["Label"].str.replace(r"[^\x00-\x7F]+", "-", regex=True).str.strip()

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Read Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
Read Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
Read Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
Read Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
Read Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
Read Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows
Read Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows
Read Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows

Shape: 2,830,743 rows × 79 columns


## 2. Gộp 15 nhãn thành 9 lớp

| Nhãn gốc | Lớp |
|---|---|
| DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest | `DoS` |
| FTP-Patator, SSH-Patator | `Brute Force` |
| Web Attack - Brute Force, Web Attack - XSS, Web Attack - Sql Injection | `Web Attack` |
| Các nhãn còn lại | Giữ nguyên |

Các nhãn được gộp thuộc cùng một họ tấn công; tách riêng thì có kiểu chỉ vài chục mẫu (Web Attack - Sql Injection: 21).

Gộp trước bước 5 và 6 vì hai bước đó so sánh theo nhãn: hai flow giống hệt nhau mang nhãn `DoS Hulk` và `DoS GoldenEye` là bản trùng cần xóa, không phải mâu thuẫn.

In [3]:
LABEL_MAP = {
    "DoS Hulk": "DoS", "DoS GoldenEye": "DoS", "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS",
    "FTP-Patator": "Brute Force", "SSH-Patator": "Brute Force",
    "Web Attack - Brute Force": "Web Attack", "Web Attack - XSS": "Web Attack",
    "Web Attack - Sql Injection": "Web Attack",
}
CLASS_ORDER = ["BENIGN", "DoS", "PortScan", "DDoS", "Brute Force", "Web Attack", "Bot", "Infiltration", "Heartbleed"]

df["Label"] = df["Label"].replace(LABEL_MAP)
class_counts["Raw"] = df["Label"].value_counts()
class_counts["Raw"].reindex(CLASS_ORDER)

Label
BENIGN          2273097
DoS              252661
PortScan         158930
DDoS             128027
Brute Force       13835
Web Attack         2180
Bot                1966
Infiltration         36
Heartbleed           11
Name: count, dtype: int64

## 3. Bỏ cột không dùng

| Cột | Lý do |
|---|---|
| `Fwd Header Length.1` | Bản sao của `Fwd Header Length` |
| 8 cột hằng số | Toàn số 0, không mang thông tin; danh sách tính bằng code |
| `Destination Port` | Cổng phản ánh cách dựng kịch bản tấn công trong lab (toàn bộ Heartbleed và Infiltration ở cổng 444, SSH-Patator ở cổng 22), không phải hành vi của flow. Giữ lại thì mô hình học thuộc số cổng |

In [4]:
constant_cols = [c for c in df.columns.drop("Label") if df[c].nunique(dropna=False) <= 1]
dropped_cols = ["Fwd Header Length.1", "Destination Port"] + constant_cols
df = df.drop(columns=dropped_cols)

FEATURES = df.columns.drop("Label").tolist()
print(f"Dropped {len(dropped_cols)} columns: {dropped_cols}")
print(f"Remaining: {len(FEATURES)} features + Label column")

Dropped 10 columns: ['Fwd Header Length.1', 'Destination Port', 'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Remaining: 68 features + Label column


## 4. Xóa dòng có NaN, Infinity

2.867 dòng (0,1%), đều là flow có `Flow Duration` = 0: hai đặc trưng tốc độ chia cho thời lượng nên ra 0/0 (NaN) hoặc x/0 (Infinity). Xóa thay vì điền giá trị, vì không có tốc độ đúng nào cho một flow dài 0 giây. `dropna()` không bắt Infinity nên đổi Infinity thành NaN trước.

In [5]:
n_before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(f"Removed {n_before - len(df):,} rows, {len(df):,} rows left")
class_counts["After NaN/Inf removal"] = df["Label"].value_counts()

Removed 2,867 rows, 2,827,876 rows left


## 5. Xóa dòng trùng

Hai dòng trùng khi giống hệt nhau ở cả 68 đặc trưng và nhãn. Bản trùng bị chia vào cả train lẫn test sẽ làm điểm test cao giả tạo.

Hai điều kiện để xóa trùng đúng:
- **Đã bỏ `Destination Port`.** Các flow PortScan chỉ khác nhau ở số cổng. Giữ cột này thì PortScan còn 90.694 mẫu sau xóa trùng, bỏ đi thì còn 1.956; phần chênh là các flow giống hệt nhau trừ số cổng, sẽ nằm ở cả hai tập.
- **Đã ở dạng `float32`.** File Monday ghi số thực nhiều chữ số hơn 7 file còn lại (`17391.3043478261` so với `17391.30435`). So ở `float64` thì cùng một flow không được coi là trùng; ở `float32` gộp thêm được 33.951 nhóm.

In [6]:
n_before = len(df)
df = df.drop_duplicates(subset=FEATURES + ["Label"])
print(f"Removed {n_before - len(df):,} duplicate rows, {len(df):,} rows left")
class_counts["After deduplication"] = df["Label"].value_counts()

Removed 628,663 duplicate rows, 2,199,213 rows left


## 6. Bỏ dòng mâu thuẫn nhãn

Sau bước 5, các dòng còn giống hệt nhau về đặc trưng chắc chắn khác nhãn. Không mô hình nào tách được chúng; đây là dấu hiệu lỗi gán nhãn đã được ghi nhận ở CIC-IDS2017 (Engelen et al., 2021). Bỏ cả nhóm (`keep=False`) vì không có căn cứ để chọn nhãn nào đúng.

In [7]:
conflicts = df.duplicated(subset=FEATURES, keep=False)
n_groups = df.loc[conflicts, FEATURES].drop_duplicates().shape[0]
print(f"{n_groups} conflicting groups, {conflicts.sum()} rows in total. Labels of those rows:")
print(df.loc[conflicts, "Label"].value_counts().to_string())

df = df[~conflicts]
class_counts["After conflict removal"] = df["Label"].value_counts()

309 conflicting groups, 618 rows in total. Labels of those rows:
Label
BENIGN      309
DoS         160
PortScan    106
Bot          40
DDoS          3


## 7. Giữ ngẫu nhiên 20% BENIGN

Còn khoảng 1,86 triệu flow BENIGN. Giữ 20% (372.482) để KNN và SMOTE chạy được trên laptop; BENIGN vẫn nhiều gấp khoảng 34.000 lần Heartbleed. Giữ toàn bộ mẫu tấn công.

Lấy mẫu trước khi chia để train và test cùng phân phối lớp. Hệ quả: tỉ lệ BENIGN trong test thấp hơn thực tế, nên precision của các lớp tấn công đo được sẽ lạc quan hơn khi triển khai.

In [8]:
benign = df[df["Label"] == "BENIGN"].sample(frac=0.2, random_state=RANDOM_STATE)
attacks = df[df["Label"] != "BENIGN"]
df = pd.concat([benign, attacks])

print(f"Kept {len(benign):,} BENIGN + {len(attacks):,} attack samples = {len(df):,} rows")
class_counts["After keeping 20% BENIGN"] = df["Label"].value_counts()

Kept 372,482 BENIGN + 336,183 attack samples = 708,665 rows


## 8. Chia train/test 80/20

- `stratify=y`: mỗi lớp được chia đúng tỉ lệ 80/20. Không phân tầng thì Heartbleed (11 mẫu) có khoảng 9% khả năng không còn mẫu nào trong test.
- Chỉ chia một lần rồi lưu lại, để mọi cấu hình được đánh giá trên cùng một tập test.

In [9]:
X = df[FEATURES]
y = df["Label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
class_counts["Train"] = y_train.value_counts()
class_counts["Test"] = y_test.value_counts()

Train: 566,932 rows × 68 features
Test:  141,733 rows × 68 features


## 9. Kiểm tra

Sai điều kiện nào thì `assert` dừng notebook:
1. Không còn NaN.
2. Không dòng test nào giống hệt một dòng train về đặc trưng (so bằng hash từng dòng). Đây là kiểm tra trực tiếp rằng bước 5–6 đã chặn rò rỉ.
3. Cả 9 lớp có mặt ở cả hai tập.

In [10]:
assert not X_train.isna().any().any() and not X_test.isna().any().any()

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False))
test_hashes = pd.util.hash_pandas_object(X_test, index=False)
n_overlap = int(test_hashes.isin(train_hashes).sum())
print(f"Test rows identical to a train row: {n_overlap}")
assert n_overlap == 0

assert set(y_train.unique()) == set(y_test.unique()) == set(CLASS_ORDER)
print("All 3 checks passed")

Test rows identical to a train row: 0
All 3 checks passed


### Số mẫu qua từng bước

In [11]:
summary = pd.DataFrame(class_counts).reindex(CLASS_ORDER).fillna(0).astype(int)
summary.loc["Total"] = summary.sum()
summary

,Raw,After NaN/Inf removal,After deduplication,After conflict removal,After keeping 20% BENIGN,Train,Test
Label,,,,,,,
BENIGN,2273097,2271320,1862721,1862412,372482,297985,74497
DoS,252661,251712,193745,193585,193585,154868,38717
PortScan,158930,158804,1956,1850,1850,1480,370
DDoS,128027,128025,128014,128011,128011,102409,25602
Brute Force,13835,13832,9150,9150,9150,7320,1830
Web Attack,2180,2180,2143,2143,2143,1714,429
Bot,1966,1956,1437,1397,1397,1118,279
Infiltration,36,36,36,36,36,29,7
Heartbleed,11,11,11,11,11,9,2


## 10. Lưu

Dạng pickle: đọc ghi nhanh, giữ nguyên kiểu `float32`. Các notebook sau đọc bằng `pd.read_pickle`.

In [12]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
X_train.assign(Label=y_train).to_pickle(OUT_DIR / "train.pkl")
X_test.assign(Label=y_test).to_pickle(OUT_DIR / "test.pkl")

for f in sorted(OUT_DIR.glob("*.pkl")):
    print(f"{f.stat().st_size / 1e6:6.1f} MB  {f}")

  64.9 MB  ../data/processed/test.pkl
 259.7 MB  ../data/processed/train.pkl
